# 2.3 Обучение binary-моделей POS

In [1]:
from pathlib import Path
import json

# Название эксперимента
EXPERIMENT_NAME = "202610_dynamic_rub_filtered"

# Периоды обучения
TRAIN_START = "2025-09-01"
TRAIN_END_EXCLUSIVE = "2026-10-01"
VALID_MONTH = "2026-10-01"
TEST_MONTH = None
INFERENCE_MONTH = "2026-12-01"

# Ищем существующий запуск с такими параметрами
EXPERIMENT_DIR = (
    Path("/home/jovyan/work/loyalty_handoff/dynamic")
    / "experiment_outputs"
    / EXPERIMENT_NAME
)

matches = []

for config_path in EXPERIMENT_DIR.glob("*/run_config.json"):
    cfg = json.loads(config_path.read_text(encoding="utf-8"))

    if (
        cfg["train_start"] == TRAIN_START
        and cfg["train_end_exclusive"] == TRAIN_END_EXCLUSIVE
        and cfg["valid_month"] == VALID_MONTH
        and cfg["test_month"] == TEST_MONTH
    ):
        matches.append((config_path.parent, cfg))

assert len(matches) == 1, (
    f"Найдено запусков с заданными периодами: {len(matches)}. "
    "Необходимо однозначно определить RUN_DIR."
)

RUN_DIR, run_config = matches[0]
RUN_TAG = run_config["run_tag"]

print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("RUN_TAG:", RUN_TAG)
print("RUN_DIR:", RUN_DIR)
print("TRAIN_START:", TRAIN_START)
print("TRAIN_END_EXCLUSIVE:", TRAIN_END_EXCLUSIVE)
print("VALID_MONTH:", VALID_MONTH)

EXPERIMENT_NAME: 202610_dynamic_rub_filtered
RUN_TAG: train_202509_202609__valid_202610__test_none__20261122_075913_858880
RUN_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880
TRAIN_START: 2025-09-01
TRAIN_END_EXCLUSIVE: 2026-10-01
VALID_MONTH: 2026-10-01


In [2]:
# PROJECT_ROOT — ветка dynamic
from pathlib import Path
import sys
import os

_branch = "dynamic"
_cwd = Path.cwd().resolve()

_candidates = [
    _cwd,
    _cwd / _branch,
    _cwd.parent / _branch
]

PROJECT_ROOT = next(
    (
        x for x in _candidates
        if x.name == _branch
        and (x / "srcc" / "nbc_project").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Не найдена папка {_branch}/srcc/nbc_project"
    )

SRC_PATH = PROJECT_ROOT / "srcc"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.chdir(PROJECT_ROOT)

# Проверяем, что используем существующий эксперимент
assert RUN_DIR.is_dir()
assert RUN_DIR.parent == (
    PROJECT_ROOT / "experiment_outputs" / EXPERIMENT_NAME
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXPERIMENT_NAME:", EXPERIMENT_NAME)
print("RUN_TAG:", RUN_TAG)
print("RUN_DIR:", RUN_DIR)

PROJECT_ROOT: /home/jovyan/work/loyalty_handoff/dynamic
EXPERIMENT_NAME: 202610_dynamic_rub_filtered
RUN_TAG: train_202509_202609__valid_202610__test_none__20261122_075913_858880
RUN_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880


In [3]:
import json
from pathlib import Path
import pandas as pd

from nbc_project.training_workflow import (
    TrainJobConfig,
    run_jobs,
    summarize_results,
    run_optional_rolling,
    run_optional_visuals,
)
from nbc_project.feature_lists import get_feature_list


with open(PROJECT_ROOT / 'configs' / 'training_jobs.json', encoding='utf-8') as f:
    TRAINING_CONFIG = json.load(f)

DEFAULT_SPLIT = TRAINING_CONFIG['default_split']
DEFAULT_SPLIT

{'train_end_exclusive': '2026-10-01',
 'valid_month': '2026-10-01',
 'test_month': None,
 'month_col': 'forecast_month',
 'add_time_features': True,
 'split_suffix': '202610_dynamic',
 'legacy_test_suffix': None,
 'iterations': 1000,
 'depth': 6,
 'l2_leaf_reg': 5.0,
 'random_seed': 42,
 'early_stopping_rounds': 50,
 'auto_class_weights': None}

## Конфигурация обучения

In [4]:
# Конфигурации двух моделей: Treatment и Control
configs = [
    TrainJobConfig(**{**DEFAULT_SPLIT, **cfg})
    for cfg in TRAINING_CONFIG["pos_binary"]
]

# Настройки текущего эксперимента
for cfg in configs:
    cfg.train_end_exclusive = TRAIN_END_EXCLUSIVE
    cfg.valid_month = VALID_MONTH
    cfg.test_month = TEST_MONTH

    cfg.split_suffix = RUN_TAG
    cfg.legacy_test_suffix = None

    cfg.output_split_dir = str(
        RUN_DIR / "splits" / "binary_bin"
    )

    cfg.model_output_dir = str(
        RUN_DIR / "models" / "binary_bin"
    )

    segment = "T" if cfg.model_key.endswith("_T") else "C"

    # Сохраняем схему названий коллеги
    cfg.model_filename = f"model_{segment}_202610_dynamic.cbm"

for cfg in configs:
    print("Модель:", cfg.model_key)
    print("Таргет:", cfg.target_col)
    print("Train end:", cfg.train_end_exclusive)
    print("Validation:", cfg.valid_month)
    print("Model dir:", cfg.model_output_dir)
    print("Model file:", cfg.model_filename)
    print("-" * 60)

Модель: binary_T
Таргет: target_flag
Train end: 2026-10-01
Validation: 2026-10-01
Model dir: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/binary_bin
Model file: model_T_202610_dynamic.cbm
------------------------------------------------------------
Модель: binary_C
Таргет: target_flag
Train end: 2026-10-01
Validation: 2026-10-01
Model dir: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/binary_bin
Model file: model_C_202610_dynamic.cbm
------------------------------------------------------------


### Опционально: dynamic feature selection — v6

- `USE_DYNAMIC_FS = False`: **current** остаётся на старом static feature set (`acrm.common_segmentation`).
- `USE_DYNAMIC_FS = True`: модель читает отдельный **full modular** dataset (`feature_store_inc_modular`) и запускает выбранный dynamic FS.
  
Dynamic FS сначала работает на sample, а затем полный parquet читается только по выбранным колонкам.


In [5]:
USE_DYNAMIC_FS = True

# Оригинальные методы Feature Selection коллеги
FS_BY_MODEL = {
    "binary_T": "psi_importance_auto_n",
    "binary_C": "psi_importance_auto_n"
}

FS_REPORT_DIR = str(
    RUN_DIR / "reports" / "feature_selection" / "binary_bin"
)

DYNAMIC_FS_PRESETS = {
    "psi_importance_auto_n": {
        "method": "psi_importance",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 200,
        "report_dir": FS_REPORT_DIR,
    },
    "psi_rfe_loss_auto_n": {
        "method": "psi_rfe",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "rfe_algorithm": "loss_function_change",
        "rfe_steps": 5,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 180,
        "report_dir": FS_REPORT_DIR,
    },
}

CORRELATION_PREFILTER = {
    "drop_correlated_features": True,
    "correlation_threshold": 0.95,
    "correlation_method": "pearson",
    "correlation_max_rows": 20_000,
    "correlation_min_unique_values": 20,
}

for _cfg in DYNAMIC_FS_PRESETS.values():
    _cfg.update(CORRELATION_PREFILTER)

PROTECTED_OFFER_FEATURES = [
    c
    for c in get_feature_list("data_fm_must_have_features")
    if c.startswith("offered_percent_cat_")
]

if not PROTECTED_OFFER_FEATURES:
    PROTECTED_OFFER_FEATURES = [
        c
        for c in get_feature_list("must_have_offered_percent_features")
        if c.startswith("offered_percent_cat_")
    ]

def make_fs_config(model_key):
    preset = FS_BY_MODEL[model_key]
    cfg = DYNAMIC_FS_PRESETS[preset].copy()
    cfg["protected_features"] = PROTECTED_OFFER_FEATURES.copy()
    return cfg


# Новые датасеты RUB Filtered
COMMON_DATA_DIR = PROJECT_ROOT.parent / "common" / "data" / "datasets"

FULL_DATA_BY_SEGMENT = {
    "T": str(
        COMMON_DATA_DIR
        / "mae_nbc_fm_loyalty_dataset_modular_full_202610_rub_filtered"
    ),
    "C": str(
        COMMON_DATA_DIR
        / "mae_nbc_fm_noloyalty_dataset_modular_full_202610_rub_filtered"
    ),
}

# Привязываем источники и Feature Selection к моделям
for cfg in configs:
    segment = "T" if cfg.model_key.endswith("_T") else "C"
    preset = FS_BY_MODEL[cfg.model_key]

    cfg.input_path = FULL_DATA_BY_SEGMENT[segment]

    assert Path(cfg.input_path).is_dir(), (
        f"Не найден датасет: {cfg.input_path}"
    )

    cfg.ignore_cols = list(
        dict.fromkeys(cfg.ignore_cols + ["period_dt", "n_treat", "rn"])
    )

    cfg.feature_selection = make_fs_config(cfg.model_key)

    cfg.model_filename = cfg.model_filename.replace(
        ".cbm",
        f"_{preset}.cbm",
    )

print("USE_DYNAMIC_FS:", USE_DYNAMIC_FS)
print("protected offered_percent:", len(PROTECTED_OFFER_FEATURES))
print("FS_REPORT_DIR:", FS_REPORT_DIR)

for cfg in configs:
    print()
    print("Модель:", cfg.model_key)
    print("Источник:", cfg.input_path)
    print("Feature Selection:", FS_BY_MODEL[cfg.model_key])
    print("Имя модели:", cfg.model_filename)

USE_DYNAMIC_FS: True
protected offered_percent: 41
FS_REPORT_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/feature_selection/binary_bin

Модель: binary_T
Источник: /home/jovyan/work/loyalty_handoff/common/data/datasets/mae_nbc_fm_loyalty_dataset_modular_full_202610_rub_filtered
Feature Selection: psi_importance_auto_n
Имя модели: model_T_202610_dynamic_psi_importance_auto_n.cbm

Модель: binary_C
Источник: /home/jovyan/work/loyalty_handoff/common/data/datasets/mae_nbc_fm_noloyalty_dataset_modular_full_202610_rub_filtered
Feature Selection: psi_importance_auto_n
Имя модели: model_C_202610_dynamic_psi_importance_auto_n.cbm


In [7]:
# Датасеты уже скачаны в common/data/datasets.
# Повторная выгрузка из HDFS не требуется.

FAST_TEST_MODE = False

for cfg in configs:
    data_path = Path(cfg.input_path)
    model_path = Path(cfg.model_output_dir) / cfg.model_filename

    assert data_path.is_dir(), f"Не найден датасет: {data_path}"

    # HDFS-файлы могут не иметь расширения .parquet
    data_files = [
        f for f in data_path.rglob("*")
        if f.is_file() and not f.name.startswith(("_", "."))
    ]

    assert data_files, f"Нет файлов данных: {data_path}"

    # Защита от перезаписи моделей
    assert not model_path.exists(), (
        f"Модель уже существует: {model_path}"
    )

    print(f"Модель: {cfg.model_key}")
    print(f"Файлов данных: {len(data_files)}")
    print(f"Максимум итераций: {cfg.iterations}")
    print(f"Early Stopping: {cfg.early_stopping_rounds}")
    print(f"Путь модели: {model_path}")
    print("-" * 60)

Модель: binary_T
Файлов данных: 22
Максимум итераций: 1000
Early Stopping: 50
Путь модели: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/binary_bin/model_T_202610_dynamic_psi_importance_auto_n.cbm
------------------------------------------------------------
Модель: binary_C
Файлов данных: 33
Максимум итераций: 1000
Early Stopping: 50
Путь модели: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/binary_bin/model_C_202610_dynamic_psi_importance_auto_n.cbm
------------------------------------------------------------


## Обучение T/C-сегментов

In [8]:
results = run_jobs(configs)


===== binary_T =====
Читаю данные: /home/jovyan/work/loyalty_handoff/common/data/datasets/mae_nbc_fm_loyalty_dataset_modular_full_202610_rub_filtered
dynamic feature selection: method=psi_importance, sample_rows=150,000, top_n=200, psi_threshold=0.2
feature-selection sample shape: (150000, 1351)
selected features: 280 / 1353
feature selection artifacts: {'feature_selection_report': '/home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/feature_selection/binary_bin/binary_T/psi_importance/feature_selection_report.csv', 'feature_selection_metadata': '/home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/feature_selection/binary_bin/binary_T/psi_importance/feature_selection_metadata.json', 'selected_features': '/home/jovyan/work/loyalty_handoff/dynamic/experiment_outp

## Сводка метрик

In [9]:
summary = summarize_results(results)
summary

,model_key,split,rows,positive_rate,avg_score,precision@0.5,recall@0.5,f1@0.5,roc_auc,pr_auc,logloss,brier,tn,fp,fn,tp
0,binary_T,valid,109819.0,0.485371,0.485371,0.744466,0.662552,0.701125,0.811531,0.822099,0.506205,0.173946,44394.0,12122.0,17987.0,35316.0
1,binary_T,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,binary_C,valid,219638.0,0.373856,0.373856,0.786297,0.623068,0.695230,0.865573,0.827428,0.413018,0.136730,123620.0,13905.0,30951.0,51162.0
3,binary_C,test,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Визуализации по таргетам, моделям, метрикам и feature importance

Блок строит презентационные графики для каждой модели/сегмента: target distribution, boxplot таргета, metrics bar, диагностику prediction/fact или ROC/PR/calibration, confusion matrix для binary и feature importance.

In [10]:
RUN_VISUALS = True

VISUALS_REPORT_DIR = str(
    RUN_DIR / "reports" / "training_visuals"
)

if RUN_VISUALS:
    visuals = run_optional_visuals(
        results,
        output_dir=VISUALS_REPORT_DIR,
    )

    print("Графики сохранены:", VISUALS_REPORT_DIR)

Интерактивные Plotly-графики сохранены в: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/training_visuals/binary_T
Интерактивные Plotly-графики сохранены в: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/training_visuals/binary_C
Графики сохранены: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/reports/training_visuals


## 5. Rolling train / walk-forward backtest

In [11]:
RUN_ROLLING_BACKTEST = False

ROLLING_MODEL_PARAMS = {
    'iterations': 300,
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 5,
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 100,
}

if RUN_ROLLING_BACKTEST:
    rolling = run_optional_rolling(
        results,
        output_dir='reports/rolling_backtest',
        min_train_months=3,
        model_params=ROLLING_MODEL_PARAMS,
    )
    rolling


In [12]:
import gc

# Освобождаем обучающие выборки, модели и результаты
del results

# Освобождаем объекты визуализаций
if "visuals" in globals():
    del visuals

collected = gc.collect()

print("Объектов собрано:", collected)
print("POS Binary: обучение завершено, память очищена")

Объектов собрано: 7431
POS Binary: обучение завершено, память очищена


## 6. Итоговые артефакты

После Run All будут сохранены:

- модели и metadata в директории из `model_output_dir`;
- split-файлы train/valid/test;
- `*_feature_importance.csv`;
- графики в `reports/training_visuals/<model_key>/`;
- rolling metrics/heatmap в `reports/rolling_backtest/<model_key>/`.

In [ ]:
{
    'summary': summary,
    'visuals_dir': 'reports/training_visuals',
    'rolling_dir': 'reports/rolling_backtest' if RUN_ROLLING_BACKTEST else None,
}
